# Distributed Spam Detection
Krzysztof Kopel, 2026

In this notebook I will try to construct (and later, deploy) an ML model to distinguish spam and normal emails. The model will be created in Ray, a library supporting distributed computing.

In [1]:
import ray
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1" # Only legacy Keras works well with Ray

In [2]:
if ray.is_initialized:
    ray.shutdown()

ray.init()

2026-06-02 16:36:55,615	ERROR services.py:1405 -- Failed to start the dashboard , return code 1
2026-06-02 16:36:55,618	ERROR services.py:1430 -- Error should be written to 'dashboard.log' or 'dashboard.err'. We are printing the last 20 lines for you. See 'https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#logging-directory-structure' to find where the log file is.
2026-06-02 16:36:55,698	ERROR services.py:1440 -- Couldn't read dashboard.log file. Error: 'utf-8' codec can't decode byte 0xb3 in position 43: invalid start byte. It means the dashboard is broken even before it initializes the logger (mostly dependency issues). Reading the dashboard.err file which contains stdout/stderr.
2026-06-02 16:36:55,737	ERROR services.py:1474 -- 
The last 20 lines of C:\Users\krzys\AppData\Local\Temp\ray\session_2026-06-02_16-36-35_716074_15368\logs\dashboard.err (it contains the error message from the dashboard): 
    raise e
  File "C:\Users\krzys\Programy_Studia\Di

Python version:,3.12.10
Ray version:,2.55.1


## Data loading and preprocessing

I will use Enron Spam dataset.

In [3]:
import pandas as pd

local_path = r"data\enron_spam_data.csv"

pdf = pd.read_csv(
    local_path,
    on_bad_lines='skip',
    encoding='utf-8',
    low_memory=False
)

pdf = pdf.fillna({"Message": "", "Subject": ""})

df = ray.data.from_pandas(pdf)
df.show(5)

2026-06-02 16:37:34,164	INFO dataset.py:3818 -- Tip: Use `take_batch()` instead of `take() / show()` to return records in pandas or numpy batch format.
2026-06-02 16:37:34,244	INFO logging.py:416 -- Registered dataset logger for dataset dataset_1_0
2026-06-02 16:37:34,431	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_1_0. Full logs are in C:\Users\krzys\AppData\Local\Temp\ray\session_2026-06-02_16-36-35_716074_15368\logs\ray-data
2026-06-02 16:37:34,433	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_1_0: InputDataBuffer[Input] -> LimitOperator[limit=5]
2026-06-02 16:37:34,444	WARNING resource_manager.py:169 -- ⚠️  Ray's object store is configured to use only 42.9% of available memory (0.6GiB out of 1.4GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.init() or by setting the RAY_DEFAULT_OBJE

{'Message ID': 0, 'Subject': 'christmas tree farm pictures', 'Message': '', 'Spam/Ham': 'ham', 'Date': '1999-12-10'}
{'Message ID': 1, 'Subject': 'vastar resources , inc .', 'Message': 'gary , production from the high island larger block a - 1 # 2 commenced on\nsaturday at 2 : 00 p . m . at about 6 , 500 gross . carlos expects between 9 , 500 and\n10 , 000 gross for tomorrow . vastar owns 68 % of the gross production .\ngeorge x 3 - 6992\n- - - - - - - - - - - - - - - - - - - - - - forwarded by george weissman / hou / ect on 12 / 13 / 99 10 : 16\nam - - - - - - - - - - - - - - - - - - - - - - - - - - -\ndaren j farmer\n12 / 10 / 99 10 : 38 am\nto : carlos j rodriguez / hou / ect @ ect\ncc : george weissman / hou / ect @ ect , melissa graves / hou / ect @ ect\nsubject : vastar resources , inc .\ncarlos ,\nplease call linda and get everything set up .\ni \' m going to estimate 4 , 500 coming up tomorrow , with a 2 , 000 increase each\nfollowing day based on my conversations with bill fis

In [ ]:
def preprocess_batch(batch):
    subject_clean = batch["Subject"].fillna("")
    message_clean = batch["Message"].fillna("")

    batch["Text"] = subject_clean + " " + message_clean

    batch["Spam/Ham"] = batch["Spam/Ham"].map({"ham": 0, "spam": 1})

    return batch

processed_df = df.map_batches(preprocess_batch, batch_format="pandas")

train, test = processed_df.train_test_split(test_size=0.2, shuffle=True)
print("Train set:")
train.show(1)
print("Test set:")
test.show(1)

## Training a model

In [5]:
import tensorflow as tf

def build_model() -> tf.keras.Sequential:

    return tf.keras.Sequential([
        tf.keras.Input(shape=(200,), dtype=tf.int32),
        tf.keras.layers.Embedding(input_dim=10000, output_dim=32),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dense(100, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

In [6]:
from ray.train.tensorflow.keras import ReportCheckpointCallback

def training_loop(config: dict):
    tf.keras.backend.clear_session()
    strategy = tf.distribute.MultiWorkerMirroredStrategy()

    batch_size = config.get("batch_size", 32)
    epochs = config.get("epochs", 4)
    train = ray.train.get_dataset_shard("train").to_tf(feature_columns="Text", label_columns="Spam/Ham", batch_size=batch_size)
    test = ray.train.get_dataset_shard("test").to_tf(feature_columns="Text", label_columns="Spam/Ham", batch_size=batch_size)

    with strategy.scope():
        model = build_model()
        model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

    model.fit(train, validation_data=test, batch_size=batch_size, epochs=epochs, callbacks=[ReportCheckpointCallback()])


In [7]:
global_vectorizer = tf.keras.layers.TextVectorization(max_tokens=10000, output_sequence_length=200)
global_vectorizer.adapt(train.to_pandas()["Text"])
vocab = global_vectorizer.get_vocabulary()

def tokenize_batch(batch):
    local_vectorizer = tf.keras.layers.TextVectorization(max_tokens=10000, output_sequence_length=200)
    local_vectorizer.set_vocabulary(vocab)
    tokenized_text = local_vectorizer(batch["Text"]).numpy()
    batch["Text"] = tokenized_text
    return batch

train = train.map_batches(tokenize_batch, batch_format="numpy")
test = test.map_batches(tokenize_batch, batch_format="numpy")

2026-06-02 16:38:01,168	INFO logging.py:416 -- Registered dataset logger for dataset dataset_6_3
2026-06-02 16:38:01,197	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_6_3 =======
2026-06-02 16:38:01,203	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-06-02 16:38:01,207	INFO logging_progress.py:227 -- Active & requested resources: 0/12 CPU, 0.0B/305.1MiB object store
2026-06-02 16:38:01,212	INFO logging_progress.py:192 -- ============================================
2026-06-02 16:38:01,225	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_6_3 execution finished in 1.93 seconds


In [10]:
from ray.train.tensorflow import TensorflowTrainer
from ray.train import RunConfig

scaling_config = ray.train.ScalingConfig(num_workers=2, use_gpu=False)

trainer = TensorflowTrainer(
    train_loop_per_worker=training_loop,
    train_loop_config={"batch_size": 32, "epochs": 5},
    scaling_config=scaling_config,
    run_config=RunConfig(storage_path=os.path.abspath("./artifacts")),
    datasets={"train": train, "test": test}
)
results = trainer.fit()

(TrainController pid=21984) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(TrainController pid=21984) I0000 00:00:1780411696.557827   28464 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
(TrainController pid=21984) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(TrainController pid=21984) I0000 00:00:1780411699.061193   28464 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
(TrainController pid=21984) WARNING:tensorflow:From C:\Users\krzys\Programy_Studia\Distributed_Spam_Detector\.venv\Lib\site-packages\tf_keras\src\losses

(pid=21108) Running Dataset dataset_33_0.: 0.00 row [00:00, ? row/s]

(pid=21108) - MapBatches(tokenize_batch):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=21108) - limit=1:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(SplitCoordinator pid=21108) ✔️  Dataset dataset_33_0 execution finished in 2.39 seconds
(RayTrainWorker pid=20784) WARNING:tensorflow:From C:\Users\krzys\Programy_Studia\Distributed_Spam_Detector\.venv\Lib\site-packages\tf_keras\src\backend.py:277: The name tf.reset_default_graph is deprecated. Please use tf.compat.v1.reset_default_graph instead.
(RayTrainWorker pid=20784) From C:\Users\krzys\Programy_Studia\Distributed_Spam_Detector\.venv\Lib\site-packages\tf_keras\src\backend.py:277: The name tf.reset_default_graph is deprecated. Please use tf.compat.v1.reset_default_graph instead.
(RayTrainWorker pid=20784) I0000 00:00:1780411733.273856   13000 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
(RayTrainWorker pid=20784) To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
(RayTrainWorker pid=27592) 

...

(pid=22720) - MapBatches(tokenize_batch):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=22720) - split(2, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(RayTrainWorker pid=20784) 843/844 [============================>.] - ETA: 0s - loss: 0.0168 - accuracy: 1.9940 [repeated 55x across cluster]
(RayTrainWorker pid=20784) 844/844 [==============================] - ETA: 0s - loss: 0.0168 - accuracy: 1.9940


(SplitCoordinator pid=22720) ✔️  Dataset test_30_4 execution finished in 3.61 seconds
(RayTrainWorker pid=20784) W0000 00:00:1780412211.645849   13000 dataset.cc:997] Input of GeneratorDatasetOp::Dataset will not be optimized because the dataset does not implement the AsGraphDefInternal() method needed to apply optimizations.
(RayTrainWorker pid=27592) Checkpoint successfully created at: Checkpoint(filesystem=local, path=C:/Users/krzys/Programy_Studia/Distributed_Spam_Detector/artifacts/ray_train_run-2026-06-02_16-48-12/checkpoint_2026-06-02_16-57-05.270230)
(RayTrainWorker pid=27592) Reporting training result 5: TrainingReport(checkpoint=Checkpoint(filesystem=local, path=C:/Users/krzys/Programy_Studia/Distributed_Spam_Detector/artifacts/ray_train_run-2026-06-02_16-48-12/checkpoint_2026-06-02_16-57-05.270230), metrics={'loss': 0.008401188999414444, 'accuracy': 0.9969968795776367, 'val_loss': 0.048653047531843185, 'val_accuracy': 0.9850237369537354}, validation=False)


(RayTrainWorker pid=27592) 
(RayTrainWorker pid=27592) 844/844 [==============================] - 150s 166ms/step - loss: 0.0084 - accuracy: 0.9970 - val_loss: 0.0487 - val_accuracy: 0.9850
(RayTrainWorker pid=20784) 


### Evaluation

In [11]:
print(f"Final results: {results.metrics}")

Final results: {'loss': 0.008401188999414444, 'accuracy': 0.9969968795776367, 'val_loss': 0.048653047531843185, 'val_accuracy': 0.9850237369537354}


Model seems to be working correctly and is ready for deployment.

In [12]:
with open("artifacts/vocabulary.txt", "w", encoding="utf-8") as file:
    file.write("\n".join(global_vectorizer.get_vocabulary()))

In [15]:
results.get_best_checkpoint(metric="val_loss", mode="min")

Checkpoint(filesystem=local, path=C:/Users/krzys/Programy_Studia/Distributed_Spam_Detector/artifacts/ray_train_run-2026-06-02_16-48-12/checkpoint_2026-06-02_16-52-12.933375)